In [ ]:
import os
from pathlib import Path
import sys

# Add the project root to the Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"Project root added to path: {project_root}")


from dotenv import load_dotenv

from footy_track.constants import (
    ROBOFLOW_BROADCAST_PROJECT,
    ROBOFLOW_WORKSPACE,
)
from footy_track.labelling import RoboflowClassificationHandler

load_dotenv()

PROJECT_ROOT = project_root
DATA_DIR = PROJECT_ROOT / "data"
ROBOFLOW_VERSION = 7

print(f"Data directory: {DATA_DIR}")


In [ ]:
# Download the dataset
handler = RoboflowClassificationHandler(
    workspace_name=ROBOFLOW_WORKSPACE,
    project_name=ROBOFLOW_BROADCAST_PROJECT,
)

dataset_path = handler.download_dataset(
    version_number=ROBOFLOW_VERSION,
    data_location=DATA_DIR,
)

print(f"Dataset downloaded to: {dataset_path}")


In [ ]:
# Inspect the dataset directory structure
for root, dirs, files in os.walk(dataset_path):
    print(f"Root: {root}")
    print(f"Dirs: {dirs}")
    print(f"Files: {files}")
    print("-" * 20)


In [ ]:
from footy_track.classifier import get_current_best_guess_classifier
from tqdm import tqdm

classifier = get_current_best_guess_classifier()

predictions = []
for root, dirs, files in os.walk(dataset_path):
    for file in tqdm(files):
        if file.endswith((".jpg", ".jpeg", ".png")):
            image_path = Path(root) / file
            prediction = classifier.predict_from_path(image_path)
            predictions.append(prediction)

print(f"Number of predictions: {len(predictions)}")


In [ ]:
import fiftyone as fo
from fiftyone import ViewField as F
import os
from pathlib import Path

# Create a FiftyOne dataset
dataset_name = "broadcast-classification-dataset"
if dataset_name in fo.list_datasets():
    fo.delete_dataset(dataset_name)
dataset = fo.Dataset(dataset_name)

# Manually load the dataset
for split in ["train", "valid", "test"]:
    split_dir = dataset_path / split
    for label in os.listdir(split_dir):
        label_dir = split_dir / label
        if not label_dir.is_dir():
            continue

        for img_file in label_dir.glob("*"):
            if img_file.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
                continue

            sample = fo.Sample(
                filepath=str(img_file.resolve()),
                tags=[split],
                ground_truth=fo.Classification(label=label),
            )
            dataset.add_sample(sample)


# Add predictions to the dataset
with dataset.save_context() as context:
    for prediction in predictions:
        sample = dataset[str(prediction.uri)]
        sample["predictions"] = fo.Classification(
            label=prediction.classification.label.value,
            confidence=prediction.classification.confidence,
        )
        context.save(sample)



In [ ]:
# Launch the FiftyOne app to visualize the dataset
session = fo.launch_app(dataset)


### Mismatched Predictions
Create a view to see the samples where the ground truth and predicted labels do not match.

In [ ]:
from fiftyone import ViewField as F

# Create a view to see the samples where the ground truth and predicted labels do not match
mismatched_view = dataset.match(F("ground_truth.label") != F("predictions.label"))

# Print the number of mismatched samples
print(f"Number of mismatched samples: {len(mismatched_view)}")

# Save the mismatched view
dataset.save_view("mismatched_view", mismatched_view)

# Set the session view to the mismatched view
session.view = mismatched_view


In [ ]:
import fiftyone.zoo as foz
from fiftyone import brain

# Generate embeddings
embedding_model = foz.load_zoo_model("mobilenet-v2-imagenet-torch", device="mps")
embeddings = dataset.compute_embeddings(embedding_model,num_workers = 8)

# Compute visualization
results = brain.compute_visualization(
    dataset, embeddings=embeddings, brain_key="img_viz"
)

## Get Selected Frames

After selecting frames in the FiftyOne app, run the cell below to get the filepaths of the selected frames.

In [ ]:
selected_frames = []


In [ ]:
selected_samples = session.selected
if not selected_samples:
    print("No samples selected in the FiftyOne app.")
else:
    print(f"{len(selected_samples)} samples selected.")

    # Create a view of the selected samples
    selected_view = dataset.select(selected_samples)

    # Get the filepaths of the selected samples
    selected_filepaths = selected_view.values("filepath")

    print("Filepaths of selected frames:")
    for filepath in selected_filepaths:
        selected_frames.append(filepath)


selected_filepaths = list(set(selected_filepaths))
print(len(selected_filepaths))
selected_filepaths


## Upload Selected Frames to Roboflow

Now, we'll use the `RoboflowClassificationHandler` to upload the selected frames to your Roboflow project. Make sure your `ROBOFLOW_API_KEY` is set as an environment variable.

In [ ]:
from footy_track.labelling import RoboflowClassificationHandler
from footy_track import constants

# Roboflow configuration
batch_name = "uploading incorrect for training"

# Initialize the handler
handler = RoboflowClassificationHandler(
    workspace_name=constants.ROBOFLOW_WORKSPACE,
    project_name=constants.ROBOFLOW_BROADCAST_PROJECT,
    classifier=classifier,
)

# Upload the images
if selected_filepaths:
    print(f"Uploading {len(selected_filepaths)} images to Roboflow...")
    handler.upload_images(
        image_paths=[Path(p) for p in selected_filepaths], batch_name=batch_name
    )
    print("Upload complete.")
else:
    print("No images to upload.")
